In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 61 — Web Research Agent

## Goal
Build an agent that can **search the web**, compare multiple sources,
and write a **cited answer** — like a mini research assistant.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Tool-calling agents** | LLM decides which tools to use and when |
| **Web search tool** | DuckDuckGo search via LangChain |
| **ReAct pattern** | Reason → Act → Observe → Repeat |
| **Source citation** | Agent grounds answers with search results |

## Architecture
```
User Question → LLM (qwen3.5) → [Decide: search or answer?]
                     ↓                    ↓
              DuckDuckGo Search      Final Answer
                     ↓                (with citations)
              Observe results
                     ↓
              Search again or answer
```

## Stack
- **LLM**: Ollama `qwen3.5:9b` (local, tool-calling capable)
- **Framework**: LangGraph (`create_react_agent`)
- **Tools**: DuckDuckGo web search

In [ ]:
import os, warnings
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from duckduckgo_search import DDGS

print("All imports OK")

## Step 1 — Define the Web Search Tool

We use DuckDuckGo (no API key needed) to search the web.
The `@tool` decorator tells LangChain this function is callable by the LLM.

In [ ]:
@tool
def web_search(query: str) -> str:
    """Search the web using DuckDuckGo. Returns top 5 results with title, URL, and snippet."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=5))
    if not results:
        return "No results found."
    formatted = []
    for i, r in enumerate(results, 1):
        formatted.append(f"[{i}] {r['title']}\n    URL: {r['href']}\n    {r['body']}")
    return "\n\n".join(formatted)

# Test the tool directly
print(web_search.invoke("LangGraph create_react_agent tutorial")[:500])

## Step 2 — Create the Agent

**`create_react_agent`** builds a ReAct agent that:
1. Receives a user question
2. Decides whether to call a tool or answer directly
3. Calls tools, observes results, and reasons about them
4. Repeats until it has enough info to answer

In [ ]:
# Initialize the LLM
llm = ChatOllama(model="qwen3.5:9b", temperature=0)

# System prompt that instructs the agent to cite sources
SYSTEM_PROMPT = """You are a web research assistant. Your job is to:
1. Search the web to find relevant information
2. Compare multiple sources when possible  
3. Write a clear, concise answer with citations

Rules:
- Always search before answering factual questions
- Cite sources using [1], [2], etc.
- If sources disagree, mention the discrepancy
- Keep answers concise but informative (3-5 paragraphs max)
- End with a Sources section listing URLs"""

# Create the agent
agent = create_react_agent(
    model=llm,
    tools=[web_search],
    prompt=SYSTEM_PROMPT,
)

print("Agent created with tools:", [t.name for t in [web_search]])

## Step 3 — Run the Agent

Let's ask some research questions and watch the agent
decide when to search and how to synthesize results.

In [ ]:
def ask_agent(question: str):
    """Run the agent on a question and display the trace."""
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print(f"{'='*70}")

    result = agent.invoke({"messages": [{"role": "user", "content": question}]})

    # Show agent trace
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 TOOL CALL: {tc['name']}({tc['args']})")
        elif role == "ToolMessage":
            print(f"📋 TOOL RESULT (first 200 chars): {msg.content[:200]}...")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 AGENT ANSWER:\n{msg.content}")

    return result

In [ ]:
# Question 1: Factual research
result1 = ask_agent("What is Model Context Protocol (MCP) and why is it important for AI agents?")

In [ ]:
# Question 2: Comparison research
result2 = ask_agent("Compare LangGraph and CrewAI for building multi-agent systems. What are the tradeoffs?")

In [ ]:
# Question 3: Current events / recent info
result3 = ask_agent("What are the latest developments in open-source LLMs as of 2025?")

## Step 4 — Adding a Follow-Up Search Tool

Let's add a tool for fetching specific pages when the agent
wants more detail from a particular URL.

In [ ]:
import requests
from html.parser import HTMLParser

class _TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self._text = []
        self._skip = False
    def handle_starttag(self, tag, attrs):
        if tag in ("script", "style", "meta", "link"):
            self._skip = True
    def handle_endtag(self, tag):
        if tag in ("script", "style", "meta", "link"):
            self._skip = False
    def handle_data(self, data):
        if not self._skip:
            stripped = data.strip()
            if stripped:
                self._text.append(stripped)
    def get_text(self):
        return " ".join(self._text)

@tool
def fetch_page(url: str) -> str:
    """Fetch and extract text content from a web page URL. Returns first 3000 characters."""
    try:
        resp = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        parser = _TextExtractor()
        parser.feed(resp.text)
        text = parser.get_text()
        return text[:3000] if text else "Could not extract text from page."
    except Exception as e:
        return f"Error fetching page: {e}"

# Create enhanced agent with both tools
agent_v2 = create_react_agent(
    model=llm,
    tools=[web_search, fetch_page],
    prompt=SYSTEM_PROMPT,
)

print("Enhanced agent created with tools:", ["web_search", "fetch_page"])

In [ ]:
# Test the enhanced agent — it can search AND deep-dive into pages
result4 = ask_agent.__wrapped__("What is the ReAct prompting pattern for LLM agents? Explain with an example.")
# Actually let's redefine with agent_v2
def ask_agent_v2(question: str):
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print(f"{'='*70}")
    result = agent_v2.invoke({"messages": [{"role": "user", "content": question}]})
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 TOOL CALL: {tc['name']}({tc['args']})")
        elif role == "ToolMessage":
            print(f"📋 TOOL RESULT: {msg.content[:200]}...")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 AGENT ANSWER:\n{msg.content}")
    return result

result4 = ask_agent_v2("What is the ReAct prompting pattern for LLM agents? Explain with an example.")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **@tool decorator** | Turns a Python function into an LLM-callable tool |
| **create_react_agent** | LangGraph helper that builds a ReAct loop |
| **Tool calling** | LLM outputs structured JSON to invoke tools |
| **ReAct loop** | Reason → Act (call tool) → Observe → Repeat |
| **DuckDuckGo** | Free web search, no API key needed |

## How the ReAct Loop Works
```
1. LLM receives question
2. LLM thinks: "I need to search for this"
3. LLM outputs: {tool: "web_search", args: {query: "..."}}
4. Framework executes the tool, returns results
5. LLM sees results, decides: search more or answer?
6. Eventually LLM outputs a final text answer
```

## Key Takeaway
Agents ≠ chatbots. An agent **decides** which tools to use and **when**.
The LLM is the "brain" that orchestrates the workflow.